## HASHAMP DATASTRUCTURE

Problem 1 - 
Detect duplicate customer events and find the most active customers

Production scenario

Imagine Zalando receives customer activity events from multiple services:

1. product views
2. add-to-cart events
3. wishlist events
4. purchases

Events arrive through a streaming/ingestion layer and are written into a Delta table.

Because of retries, network issues, or upstream bugs, the same event can arrive multiple times.

You need to build a transformation that:

1. Removes duplicate events based on event_id.
2. Calculates how many events each customer generated.
3. Finds customers with more than N events.
4. Handles different data types, including nested data.
5. Is efficient when the dataset becomes very large.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
    DoubleType,
    DecimalType,
    BooleanType,
    TimestampType,
    DateType,
    ArrayType,
    MapType
)

data = [

    # ---------------------------------------------------------
    # Normal events
    # ---------------------------------------------------------

    (
        "evt_001",
        "cust_001",
        "view",
        "prod_100",
        1,
        129.99,
        True,
        "2026-09-20 09:00:00",
        "2026-09-20",
        ["homepage", "recommendation"],
        {"device": "mobile", "country": "IE", "channel": "organic"}
    ),

    (
        "evt_002",
        "cust_001",
        "cart",
        "prod_100",
        2,
        129.99,
        True,
        "2026-09-20 09:02:00",
        "2026-09-20",
        ["product_page"],
        {"device": "mobile", "country": "IE", "channel": "organic"}
    ),

    (
        "evt_003",
        "cust_001",
        "purchase",
        "prod_100",
        2,
        129.99,
        True,
        "2026-09-20 09:05:00",
        "2026-09-20",
        ["checkout", "payment"],
        {"device": "mobile", "country": "IE", "channel": "organic"}
    ),

    (
        "evt_004",
        "cust_002",
        "view",
        "prod_200",
        1,
        79.99,
        False,
        "2026-09-20 10:00:00",
        "2026-09-20",
        ["search"],
        {"device": "desktop", "country": "DE", "channel": "paid"}
    ),

    (
        "evt_005",
        "cust_002",
        "purchase",
        "prod_200",
        1,
        79.99,
        True,
        "2026-09-20 10:05:00",
        "2026-09-20",
        ["checkout"],
        {"device": "desktop", "country": "DE", "channel": "paid"}
    ),

    (
        "evt_006",
        "cust_003",
        "wishlist",
        "prod_300",
        1,
        49.50,
        True,
        "2026-09-20 11:00:00",
        "2026-09-20",
        ["wishlist"],
        {"device": "tablet", "country": "FR", "channel": "organic"}
    ),

    # ---------------------------------------------------------
    # Duplicate event
    # ---------------------------------------------------------

    (
        "evt_003",
        "cust_001",
        "purchase",
        "prod_100",
        2,
        129.99,
        True,
        "2026-09-20 09:05:00",
        "2026-09-20",
        ["checkout", "payment"],
        {"device": "mobile", "country": "IE", "channel": "organic"}
    ),

    # ---------------------------------------------------------
    # Same customer, same event type, different event
    # ---------------------------------------------------------

    (
        "evt_007",
        "cust_001",
        "purchase",
        "prod_101",
        1,
        199.99,
        True,
        "2026-09-20 12:00:00",
        "2026-09-20",
        ["checkout"],
        {"device": "mobile", "country": "IE", "channel": "email"}
    ),

    (
        "evt_008",
        "cust_001",
        "purchase",
        "prod_102",
        1,
        89.99,
        True,
        "2026-09-20 12:10:00",
        "2026-09-20",
        ["checkout"],
        {"device": "mobile", "country": "IE", "channel": "email"}
    ),

    # ---------------------------------------------------------
    # Null product
    # ---------------------------------------------------------

    (
        "evt_009",
        "cust_004",
        "search",
        None,
        None,
        None,
        True,
        "2026-09-20 13:00:00",
        "2026-09-20",
        ["search"],
        {"device": "mobile", "country": "NL", "channel": "organic"}
    ),

    # ---------------------------------------------------------
    # Null customer
    # ---------------------------------------------------------

    (
        "evt_010",
        None,
        "view",
        "prod_400",
        1,
        25.00,
        False,
        "2026-09-20 13:05:00",
        "2026-09-20",
        ["recommendation"],
        {"device": "mobile", "country": "NL", "channel": "organic"}
    ),

    # ---------------------------------------------------------
    # Boolean variation
    # ---------------------------------------------------------

    (
        "evt_011",
        "cust_005",
        "view",
        "prod_500",
        1,
        399.99,
        False,
        "2026-09-20 14:00:00",
        "2026-09-20",
        ["homepage"],
        {"device": "desktop", "country": "BE", "channel": "social"}
    ),

    # ---------------------------------------------------------
    # Same timestamp - important edge case
    # ---------------------------------------------------------

    (
        "evt_012",
        "cust_005",
        "cart",
        "prod_500",
        1,
        399.99,
        True,
        "2026-09-20 14:05:00",
        "2026-09-20",
        ["product_page"],
        {"device": "desktop", "country": "BE", "channel": "social"}
    ),

    (
        "evt_013",
        "cust_005",
        "purchase",
        "prod_500",
        1,
        399.99,
        True,
        "2026-09-20 14:05:00",
        "2026-09-20",
        ["checkout"],
        {"device": "desktop", "country": "BE", "channel": "social"}
    ),

    # ---------------------------------------------------------
    # Different device
    # ---------------------------------------------------------

    (
        "evt_014",
        "cust_006",
        "view",
        "prod_600",
        1,
        59.99,
        True,
        "2026-09-20 15:00:00",
        "2026-09-20",
        ["search", "recommendation"],
        {"device": "tablet", "country": "ES", "channel": "affiliate"}
    ),

    (
        "evt_015",
        "cust_006",
        "cart",
        "prod_600",
        3,
        59.99,
        True,
        "2026-09-20 15:02:00",
        "2026-09-20",
        ["product_page"],
        {"device": "tablet", "country": "ES", "channel": "affiliate"}
    ),

    # ---------------------------------------------------------
    # Late arriving event
    # ---------------------------------------------------------

    (
        "evt_016",
        "cust_006",
        "purchase",
        "prod_600",
        2,
        59.99,
        True,
        "2026-09-19 23:59:00",
        "2026-09-19",
        ["checkout"],
        {"device": "tablet", "country": "ES", "channel": "affiliate"}
    ),

    # ---------------------------------------------------------
    # Another duplicate
    # ---------------------------------------------------------

    (
        "evt_016",
        "cust_006",
        "purchase",
        "prod_600",
        2,
        59.99,
        True,
        "2026-09-19 23:59:00",
        "2026-09-19",
        ["checkout"],
        {"device": "tablet", "country": "ES", "channel": "affiliate"}
    )
]


schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("customer_id", StringType(), True),
    StructField("event_type", StringType(), False),
    StructField("product_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("is_logged_in", BooleanType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("event_date", StringType(), True),
    StructField("tags", ArrayType(StringType()), True),
    StructField("metadata", MapType(StringType(), StringType()), True)
])


df = spark.createDataFrame(data, schema)


df = (
    df
    .withColumn(
        "event_timestamp",
        F.to_timestamp("event_timestamp")
    )
    .withColumn(
        "event_date",
        F.to_date("event_date")
    )
)


display(df)

df.printSchema()

In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.window import *
from pyspark.sql.types import *
from math import *
import builtins

Removes duplicate events based on event_id.

In [0]:
removedup_simple = df.dropDuplicates(["event_id"])

In [0]:
removedup = (
    df
    .withColumn("rn",row_number().over(Window.partitionBy("event_id").orderBy(col("event_timestamp").desc())))
    .filter(col("rn")==1)
    .drop("rn")
)

display(removedup)

Count events per customer

In [0]:
customer_activity = (
    removedup
    .groupBy("customer_id")
    .agg(
        count("*").alias("event_count")
    )
    .filter(col("event_count") > 2)
)

display(customer_activity)

For every customer, calculate total events, total quantity, average product price, number of purchases, and the timestamp of their latest activity.

In [0]:
customer_summary = (
    removedup
    .groupBy("customer_id")
    .agg(
        count("*").alias("event_count"),
        sum("quantity").alias("total_quantity"),
        avg("price").alias("avg_price"),
        sum(
            when(F.col("event_type") == "purchase", 1).otherwise(0)
        ).alias("purchase_count"),
        max("event_timestamp").alias("last_activity")
    )
)

display(customer_summary)

For every customer, tell me which device they use most frequently

In [0]:
customer_device_df = (
    df
    .filter(col("customer_id").isNotNull())
    .withColumn("device",col("metadata")["device"])
)

device_counts = (
    customer_device_df
    .groupBy(
        "customer_id",
        "device"
    )
    .count()
)

device_counts.display()


## PRODUCTION PROBLEMS

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, IntegerType,
    BooleanType, TimestampType,
    DecimalType, MapType, ArrayType
)
from datetime import datetime
from decimal import Decimal

spark = SparkSession.builder.getOrCreate()

# ============================================================
# SCHEMA
# ============================================================

item_schema = StructType([
    StructField("sku", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DecimalType(12, 2), True)
])

schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("order_id", LongType(), False),
    StructField("customer_id", LongType(), True),
    StructField("event_type", StringType(), True),

    # Business event timestamp
    StructField("event_ts", TimestampType(), True),

    # When the event reached our platform
    StructField("ingestion_ts", TimestampType(), True),

    StructField("amount", DecimalType(12, 2), True),
    StructField("currency", StringType(), True),

    StructField("is_test", BooleanType(), True),

    # Semi-structured attributes
    StructField(
        "metadata",
        MapType(StringType(), StringType()),
        True
    ),

    # Nested data
    StructField(
        "items",
        ArrayType(item_schema),
        True
    )
])


# ============================================================
# SOURCE DATA
# ============================================================

data = [

    # --------------------------------------------------------
    # ORDER 1001
    # Normal order + exact duplicate/retry
    # --------------------------------------------------------

    (
        "e-1001-01",
        1001,
        501,
        "CREATED",
        datetime(2026, 9, 20, 10, 0, 0),
        datetime(2026, 9, 20, 10, 0, 5),
        Decimal("149.95"),
        "EUR",
        False,
        {"country": "DE", "device": "mobile"},
        [
            ("SKU-101", 2, Decimal("49.99")),
            ("SKU-102", 1, Decimal("49.97"))
        ]
    ),

    (
        "e-1001-01",
        1001,
        501,
        "CREATED",
        datetime(2026, 9, 20, 10, 0, 0),
        datetime(2026, 9, 20, 10, 0, 7),
        Decimal("149.95"),
        "EUR",
        False,
        {"country": "DE", "device": "mobile"},
        [
            ("SKU-101", 2, Decimal("49.99")),
            ("SKU-102", 1, Decimal("49.97"))
        ]
    ),

    (
        "e-1001-02",
        1001,
        501,
        "PAID",
        datetime(2026, 9, 20, 10, 5, 0),
        datetime(2026, 9, 20, 10, 5, 3),
        Decimal("149.95"),
        "EUR",
        False,
        {"country": "DE", "device": "mobile"},
        [
            ("SKU-101", 2, Decimal("49.99")),
            ("SKU-102", 1, Decimal("49.97"))
        ]
    ),

    (
        "e-1001-03",
        1001,
        501,
        "SHIPPED",
        datetime(2026, 9, 20, 15, 0, 0),
        datetime(2026, 9, 20, 15, 0, 10),
        Decimal("149.95"),
        "EUR",
        False,
        {"country": "DE", "carrier": "DHL"},
        [
            ("SKU-101", 2, Decimal("49.99")),
            ("SKU-102", 1, Decimal("49.97"))
        ]
    ),


    # --------------------------------------------------------
    # ORDER 1002
    #
    # CONFLICTING DUPLICATE
    #
    # Same event_id appears with different event_type.
    # Later ingestion says CANCELLED.
    # --------------------------------------------------------

    (
        "e-1002-01",
        1002,
        502,
        "CREATED",
        datetime(2026, 9, 20, 11, 0, 0),
        datetime(2026, 9, 20, 11, 0, 2),
        Decimal("79.99"),
        "EUR",
        False,
        {"country": "FR", "device": "desktop"},
        [
            ("SKU-201", 1, Decimal("79.99"))
        ]
    ),

    (
        "e-1002-02",
        1002,
        502,
        "PAID",
        datetime(2026, 9, 20, 11, 10, 0),
        datetime(2026, 9, 20, 11, 10, 5),
        Decimal("79.99"),
        "EUR",
        False,
        {"country": "FR", "device": "desktop"},
        [
            ("SKU-201", 1, Decimal("79.99"))
        ]
    ),

    # Same event ID but conflicting payload
    (
        "e-1002-02",
        1002,
        502,
        "CANCELLED",
        datetime(2026, 9, 20, 11, 10, 0),
        datetime(2026, 9, 20, 11, 12, 0),
        Decimal("79.99"),
        "EUR",
        False,
        {"country": "FR", "device": "desktop"},
        [
            ("SKU-201", 1, Decimal("79.99"))
        ]
    ),


    # --------------------------------------------------------
    # ORDER 1003
    #
    # OUT-OF-ORDER ARRIVAL
    #
    # SHIPPED arrives before PAID from an ingestion perspective.
    # event_ts represents actual business ordering.
    # --------------------------------------------------------

    (
        "e-1003-01",
        1003,
        503,
        "CREATED",
        datetime(2026, 9, 20, 9, 0, 0),
        datetime(2026, 9, 20, 9, 0, 2),
        Decimal("249.50"),
        "GBP",
        False,
        {"country": "GB", "device": "mobile"},
        [
            ("SKU-301", 1, Decimal("199.50")),
            ("SKU-302", 1, Decimal("50.00"))
        ]
    ),

    # SHIPPED reaches platform first
    (
        "e-1003-03",
        1003,
        503,
        "SHIPPED",
        datetime(2026, 9, 20, 9, 30, 0),
        datetime(2026, 9, 20, 9, 31, 0),
        Decimal("249.50"),
        "GBP",
        False,
        {"country": "GB", "carrier": "RoyalMail"},
        [
            ("SKU-301", 1, Decimal("199.50")),
            ("SKU-302", 1, Decimal("50.00"))
        ]
    ),

    # PAID event arrives late
    (
        "e-1003-02",
        1003,
        503,
        "PAID",
        datetime(2026, 9, 20, 9, 10, 0),
        datetime(2026, 9, 20, 10, 15, 0),
        Decimal("249.50"),
        "GBP",
        False,
        {"country": "GB", "device": "mobile"},
        [
            ("SKU-301", 1, Decimal("199.50")),
            ("SKU-302", 1, Decimal("50.00"))
        ]
    ),


    # --------------------------------------------------------
    # ORDER 1004
    #
    # NULL VALUES
    # --------------------------------------------------------

    (
        "e-1004-01",
        1004,
        None,
        "CREATED",
        datetime(2026, 9, 21, 8, 0, 0),
        datetime(2026, 9, 21, 8, 0, 3),
        Decimal("59.99"),
        "EUR",
        False,
        {"country": "NL"},
        [
            ("SKU-401", 1, Decimal("59.99"))
        ]
    ),

    (
        "e-1004-02",
        1004,
        None,
        "PAID",
        datetime(2026, 9, 21, 8, 5, 0),
        datetime(2026, 9, 21, 8, 5, 5),
        Decimal("59.99"),
        "EUR",
        False,
        None,
        [
            ("SKU-401", 1, Decimal("59.99"))
        ]
    ),


    # --------------------------------------------------------
    # ORDER 1005
    #
    # TEST ORDER
    # --------------------------------------------------------

    (
        "e-1005-01",
        1005,
        505,
        "CREATED",
        datetime(2026, 9, 21, 9, 0, 0),
        datetime(2026, 9, 21, 9, 0, 1),
        Decimal("999.99"),
        "EUR",
        True,
        {"country": "DE", "environment": "test"},
        [
            ("TEST-SKU", 1, Decimal("999.99"))
        ]
    ),


    # --------------------------------------------------------
    # ORDER 1006
    #
    # DUPLICATE SKU INSIDE ARRAY
    # --------------------------------------------------------

    (
        "e-1006-01",
        1006,
        506,
        "CREATED",
        datetime(2026, 9, 21, 10, 0, 0),
        datetime(2026, 9, 21, 10, 0, 3),
        Decimal("119.97"),
        "EUR",
        False,
        {"country": "DE", "device": "app"},
        [
            ("SKU-601", 1, Decimal("39.99")),
            ("SKU-601", 2, Decimal("39.99"))
        ]
    ),


    # --------------------------------------------------------
    # ORDER 1007
    #
    # AMOUNT MISMATCH
    #
    # Item total = 100.00
    # reported amount = 120.00
    # --------------------------------------------------------

    (
        "e-1007-01",
        1007,
        507,
        "PAID",
        datetime(2026, 9, 21, 11, 0, 0),
        datetime(2026, 9, 21, 11, 0, 2),
        Decimal("120.00"),
        "EUR",
        False,
        {"country": "IT", "device": "mobile"},
        [
            ("SKU-701", 2, Decimal("50.00"))
        ]
    ),


    # --------------------------------------------------------
    # ORDER 1008
    #
    # NULL amount
    # --------------------------------------------------------

    (
        "e-1008-01",
        1008,
        508,
        "CREATED",
        datetime(2026, 9, 21, 12, 0, 0),
        datetime(2026, 9, 21, 12, 0, 5),
        None,
        "EUR",
        False,
        {"country": "ES"},
        [
            ("SKU-801", 1, Decimal("25.00"))
        ]
    ),


    # --------------------------------------------------------
    # ORDER 1009
    #
    # EVENT WITH EMPTY ITEMS ARRAY
    # --------------------------------------------------------

    (
        "e-1009-01",
        1009,
        509,
        "CANCELLED",
        datetime(2026, 9, 21, 13, 0, 0),
        datetime(2026, 9, 21, 13, 0, 1),
        Decimal("0.00"),
        "EUR",
        False,
        {"country": "DE"},
        []
    ),


    # --------------------------------------------------------
    # ORDER 1010
    #
    # Multiple lifecycle events
    # --------------------------------------------------------

    (
        "e-1010-01",
        1010,
        510,
        "CREATED",
        datetime(2026, 9, 22, 14, 0, 0),
        datetime(2026, 9, 22, 14, 0, 1),
        Decimal("199.99"),
        "EUR",
        False,
        {"country": "DE", "device": "desktop"},
        [
            ("SKU-901", 1, Decimal("199.99"))
        ]
    ),

    (
        "e-1010-02",
        1010,
        510,
        "PAID",
        datetime(2026, 9, 22, 14, 5, 0),
        datetime(2026, 9, 22, 14, 5, 1),
        Decimal("199.99"),
        "EUR",
        False,
        {"country": "DE", "device": "desktop"},
        [
            ("SKU-901", 1, Decimal("199.99"))
        ]
    ),

    (
        "e-1010-03",
        1010,
        510,
        "SHIPPED",
        datetime(2026, 9, 22, 18, 0, 0),
        datetime(2026, 9, 22, 18, 0, 2),
        Decimal("199.99"),
        "EUR",
        False,
        {"country": "DE", "carrier": "DHL"},
        [
            ("SKU-901", 1, Decimal("199.99"))
        ]
    ),

    (
        "e-1010-04",
        1010,
        510,
        "DELIVERED",
        datetime(2026, 9, 24, 12, 0, 0),
        datetime(2026, 9, 24, 12, 0, 4),
        Decimal("199.99"),
        "EUR",
        False,
        {"country": "DE", "carrier": "DHL"},
        [
            ("SKU-901", 1, Decimal("199.99"))
        ]
    ),
]


# ============================================================
# CREATE DATAFRAME
# ============================================================

orders_df = spark.createDataFrame(data, schema)

display(orders_df)

Distinguish true retries from conflicting duplicates. Don't assume every repeated event_id is an identical duplicate.


In [0]:
dedup_df = (
    orders_df
    .filter(col("is_test") == False)
    .withColumn("rn",row_number().over(Window.partitionBy("event_id").orderBy(col("ingestion_ts").desc())))
    .filter(col("rn") == 1)
    .drop(col("rn"))
)

dedup_df.display()

Find orders containing a duplicate SKU inside the items array

In [0]:
dup_sku = (
    orders_df
    .withColumn("sku",transform("items",lambda x : x["sku"]))
    .withColumn("isDuplicateSku",size("sku") != size(array_distinct("sku")))
    .filter(col("isDuplicateSku")== True)
)

dup_sku.display()

Calculate the value of every order line:

In [0]:
value_orderline = (
    orders_df
    .withColumn("totalorderval",aggregate(transform("items",lambda x:x["quantity"] * x["unit_price"]),lit(0.00),lambda acc,x: acc+x))
)

value_orderline.display()

Does any invalid item exist?

In [0]:
invalid_items_df = (
    orders_df
    .withColumn("isInvalid",exists("items",lambda x:x["sku"].isNotNull() & (x["quantity"] > 1)))
)

invalid_items_df.display()

Reconstruct an ARRAY<STRUCT>
This one is especially useful.
Instead of extracting values, you can transform one struct into another:

In [0]:
reconstruct_df = (
    orders_df
    .withColumn("lineitems",transform("items",lambda x:struct(upper(x["sku"]),(x["quantity"]*x["unit_price"]).alias("lineamount"))))
)

reconstruct_df.display()

Add array position/index
A less obvious feature: transform can give you the array index.

In [0]:
index_pos_df = (
    orders_df
    .withColumn("index",transform("items",lambda x,i: struct(i.alias("position"),x["sku"].alias("sku"))))
)

index_pos_df.display()

In [0]:
from typing import List, Tuple

from pyspark.sql import DataFrame
from pyspark.sql.column import Column
from pyspark.sql import functions as F

class ComputeCols:
    
    def __init__(self,df:DataFrame):
        self.df = df

    def add_derived_columns(self) -> "ComputeCols":
        
        self.df = self.df.withColumn("totalorderval",F.aggregate(F.transform("items",lambda x:x["quantity"]*x["unit_price"]),F.lit(0.00),lambda acc,x: acc+x))

        return self
    
    def result(self) -> DataFrame:
        """
        Return all records including validation information.
        """

        return self.df

In [0]:
# from typing import List, Tuple

# from pyspark.sql import DataFrame
# from pyspark.sql.column import Column
# from pyspark.sql import functions as F

class OrderValidator:

    def __init__(self,df:DataFrame,amount_threshold:float):
        self.df = df
        self.amount_threshold = amount_threshold
    
    def get_rules(self) -> List[Tuple[str,Column]]:

        sku = F.transform("items",lambda x:x["sku"])

        rules = [
            ("NULL CUSTOMER",F.col("customer_id").isNull()),
            ("BREACH AMOUNT",F.col("amount") > self.amount_threshold),
            ("NEGATIVE AMOUNT",F.col("amount") < 0.00),
            ("AMOUNT MISMATCH",(F.col("amount")!= F.col("totalorderval")) & (F.col("amount").isNotNull()) & (F.col("totalorderval").isNotNull())),
            ("NULL ITEMS",F.size("items")==0),
            ("INVALID QUANTITY",F.exists("items",lambda x:x["quantity"] <= 0)),
            ("NULL SKU",F.exists("items",lambda x:x["sku"].isNull())),
            ("DUPLICATE SKU",F.size(sku)!= F.size(F.array_distinct(sku)))
        ]

        return rules
    
    def validate(self) -> "OrderValidator":

        rules = self.get_rules()

        errors = [F.when(condition, F.lit(error)) for error,condition in rules]
        self.errorlist  = [error for error,condition in rules]

        self.df = self.df.withColumn("validation_errors",F.filter(F.array(*errors),lambda x:x.isNotNull()))

        return self
    
    def QuarintineRecords(self) -> "OrderValidator":

        self.df =  self.df.filter(F.size("validation_errors") > 0)

        return self

    def returnerrorlist(self) -> list[str]:
        return self.errorlist
    
    def result(self) -> DataFrame:
        """
        Return all records including validation information.
        """

        return self.df

In [0]:
compute = (
    ComputeCols(orders_df)
    .add_derived_columns()
)

compute_df = compute.result()

compute_df.show()

In [0]:
validated = OrderValidator(compute_df,50.00).validate()

validator_df = validated.result().display()

error_list = validated.returnerrorlist()
print(error_list)



In [0]:
from pyspark.sql import functions as F

result_df = orders_df.withColumn(
    "item_metrics",

    F.aggregate(

        # 1. Array being processed
        F.col("items"),

        # 2. Initial accumulator
        F.struct(
            F.lit(0).cast("decimal(12,2)").alias("total_amount"),
            F.lit(0).cast("bigint").alias("total_quantity"),
            F.lit(0).cast("bigint").alias("item_count"),
            F.lit(0).cast("bigint").alias("invalid_item_count"),
            F.lit(None).cast("decimal(12,2)").alias("max_unit_price")
        ),

        # 3. Merge each item into accumulator
        lambda acc, item: F.struct(

            (
                acc["total_amount"]
                +
                (
                    item["quantity"]
                    * item["unit_price"]
                )
            )
            .cast("decimal(12,2)")
            .alias("total_amount"),

            (
                acc["total_quantity"]
                + item["quantity"]
            )
            .cast("bigint")
            .alias("total_quantity"),

            (
                acc["item_count"]
                + F.lit(1)
            )
            .cast("bigint")
            .alias("item_count"),

            (
                acc["invalid_item_count"]
                +
                F.when(
                    item["sku"].isNull()
                    | (item["quantity"] <= 0)
                    | item["unit_price"].isNull(),
                    F.lit(1)
                ).otherwise(F.lit(0))
            )
            .cast("bigint")
            .alias("invalid_item_count"),

            F.when(
                acc["max_unit_price"].isNull(),
                item["unit_price"]
            )
            .otherwise(
                F.greatest(
                    acc["max_unit_price"],
                    item["unit_price"]
                )
            )
            .cast("decimal(12,2)")
            .alias("max_unit_price")
        )
    )
)

result_df.display()